# Construcción de un KD-Tree

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import time, math, ast, re, threading

class KDNode:
    def __init__(self, point, axis):
        self.point = point
        self.axis  = axis
        self.left  = None
        self.right = None

AXIS_NAMES  = ['X','Y','Z','W','V']
AXIS_FILL   = ['#FAC775','#9FE1CB','#B5D4F4','#F7C1C1','#CECBF6']
AXIS_STROKE = ['#BA7517','#0F6E56','#185FA5','#A32D2D','#534AB7']
AXIS_TEXT   = ['#412402','#04342C','#042C53','#501313','#26215C']

def build_kd_logged(points, depth=0, steps=None, path="raíz"):
    if steps is None: steps = []
    if not points:    return None, steps
    k    = len(points[0])
    axis = depth % k
    ax_n = AXIS_NAMES[axis] if axis < len(AXIS_NAMES) else f'd{axis}'
    spts = sorted(points, key=lambda p: p[axis])
    mid  = len(spts) // 2
    med  = spts[mid]
    steps.append({
        "path":  path, "depth": depth, "axis": axis, "axis_nm": ax_n, "k": k,
        "points": [tuple(p) for p in points],
        "sorted": [tuple(p) for p in spts],
        "median_idx": mid, "median": tuple(med),
        "left_pts":  [tuple(p) for p in spts[:mid]],
        "right_pts": [tuple(p) for p in spts[mid+1:]],
    })
    node        = KDNode(list(med), axis)
    node.left,  steps = build_kd_logged(spts[:mid],   depth+1, steps, path+" > izq")
    node.right, steps = build_kd_logged(spts[mid+1:], depth+1, steps, path+" > der")
    return node, steps

def build_partial(steps, up_to):
    if not steps: return None
    axis_map = {}
    for i in range(min(up_to + 1, len(steps))):
        s = steps[i]
        key = ','.join(str(v) for v in s['median'])
        axis_map[key] = s['axis']

    def ins(node, pt, depth):
        if node is None:
            key = ','.join(str(v) for v in pt)
            axis = axis_map.get(key, depth % steps[0]['k'])
            return KDNode(list(pt), axis)
        if pt[node.axis] < node.point[node.axis]:
            node.left  = ins(node.left,  pt, depth + 1)
        else:
            node.right = ins(node.right, pt, depth + 1)
        return node

    root = None
    for i in range(min(up_to + 1, len(steps))):
        root = ins(root, steps[i]['median'], 0)
    return root

def compute_pos(node, d=0, l=0.0, r=1.0, pos=None, edges=None):
    if pos   is None: pos   = {}
    if edges is None: edges = []
    if node  is None: return pos, edges
    mx = (l + r) / 2
    pos[id(node)] = (mx, d, node)
    if node.left:
        edges.append((id(node), id(node.left)))
        compute_pos(node.left,  d+1, l,  mx, pos, edges)
    if node.right:
        edges.append((id(node), id(node.right)))
        compute_pos(node.right, d+1, mx, r,  pos, edges)
    return pos, edges

def tree_svg(root, highlight=None, W=1000, R=32):
    if root is None:
        return ('<svg width="100%" height="80" viewBox="0 0 1000 80">'
                '<text x="500" y="45" text-anchor="middle" fill="#888" '
                'font-family="Space Mono,monospace" font-size="16">'
                'Árbol vacío</text></svg>')

    pos, edges = compute_pos(root)
    if not pos: return ''
    max_d = max(v[1] for v in pos.values())
    GAP   = 140
    H     = (max_d + 1) * GAP + 120
    sx    = lambda xn: 80 + xn * (W - 160)
    sy    = lambda d:  80 + d * GAP

    mid_id = str(id(root))[:6]
    parts = [f'<svg width="100%" height="{H}" viewBox="0 0 {W} {H}" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg">']
    parts.append(f'<defs><marker id="arr{mid_id}" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">'
                 f'<polygon points="0 0,10 5,0 10" fill="rgba(180,180,180,0.6)"/></marker></defs>')

    for pid, cid in edges:
        px, pd, _ = pos[pid]; cx, cd, _ = pos[cid]
        x1, y1 = sx(px), sy(pd)
        x2, y2 = sx(cx), sy(cd)
        dx, dy  = x2 - x1, y2 - y1
        dist    = math.sqrt(dx*dx + dy*dy) or 1
        x1s = x1 + dx/dist*(R+4); y1s = y1 + dy/dist*(R+4)
        x2e = x2 - dx/dist*(R+8); y2e = y2 - dy/dist*(R+8)
        parts.append(f'<line x1="{x1s:.1f}" y1="{y1s:.1f}" x2="{x2e:.1f}" y2="{y2e:.1f}" '
                     f'stroke="rgba(180,180,180,0.4)" stroke-width="2" marker-end="url(#arr{mid_id})"/>')

    for nid, (xn, yd, node) in pos.items():
        x, y  = sx(xn), sy(yd)
        ai    = node.axis % len(AXIS_FILL)
        fill, stroke, tc = AXIS_FILL[ai], AXIS_STROKE[ai], AXIS_TEXT[ai]
        is_hi = highlight and tuple(node.point) == tuple(highlight)
        label = ','.join(str(v) for v in node.point)
        ax_n  = AXIS_NAMES[node.axis] if node.axis < len(AXIS_NAMES) else f'd{node.axis}'

        if is_hi:
            parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{R+12}" fill="{fill}" opacity="0.2"/>'
                         f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{R+6}" fill="none" stroke="{stroke}" stroke-width="2.5" stroke-dasharray="5 3"/>')

        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{R}" fill="{fill}" stroke="{stroke}" stroke-width="{4 if is_hi else 2}"/>')
        parts.append(f'<text x="{x:.1f}" y="{y:.1f}" text-anchor="middle" dominant-baseline="central" '
                     f'font-family="Space Mono,monospace" font-size="14" font-weight="700" fill="{tc}">{label}</text>')
        parts.append(f'<text x="{x:.1f}" y="{y+R+20:.1f}" text-anchor="middle" font-family="Space Mono,monospace" '
                     f'font-size="11" font-weight="700" fill="{stroke}" text-transform="uppercase">eje {ax_n}</text>')

    parts.append('</svg>')
    return ''.join(parts)

def step_html(step):
    ai, ax = step['axis'] % len(AXIS_FILL), step['axis_nm']
    col, bg, tc = AXIS_STROKE[ai], AXIS_FILL[ai], AXIS_TEXT[ai]

    rows = ''
    for i, p in enumerate(step['sorted']):
        is_m = (i == step['median_idx'])
        row_style = f'background:{bg};' if is_m else ''
        text_style = f'font-weight:700;color:{tc};' if is_m else 'color:#bbb;'
        med_label = f'<td style="color:{col};font-weight:700;padding:8px 15px">MEDIANA</td>' if is_m else '<td></td>'
        rows += f'<tr style="{row_style}"><td style="padding:10px 20px;color:#888">{i}</td>' \
                f'<td style="padding:10px 20px;{text_style}">{p}</td>{med_label}</tr>'

    lft = str(step['left_pts']) or '∅'
    rgt = str(step['right_pts']) or '∅'

    return (
        f'<div style="font-family:Space Mono,monospace;background:#0d0d1a;border:1px solid rgba(255,255,255,0.1);border-radius:12px;padding:30px">'
        f'<div style="font-size:14px;letter-spacing:4px;color:#666;margin-bottom:20px;text-transform:uppercase;opacity:0.6">Detalle del Paso</div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:25px">'
        f'<div style="background:#07060f;border:1px solid rgba(255,255,255,0.1);border-radius:8px;padding:20px">'
        f'<div style="font-size:11px;color:#555;margin-bottom:8px;letter-spacing:1px">UBICACIÓN</div>'
        f'<div style="color:#eee;font-size:16px">{step["path"]}</div></div>'
        f'<div style="background:#07060f;border:1px solid rgba(255,255,255,0.1);border-radius:8px;padding:20px">'
        f'<div style="font-size:11px;color:#555;margin-bottom:8px;letter-spacing:1px">EJE DE PARTICIÓN</div>'
        f'<div style="font-size:16px;color:#ccc">{step["depth"]} mod {step["k"]} = <span style="color:{col};font-weight:bold;font-size:22px">{ax}</span></div></div></div>'
        f'<div style="margin-bottom:25px"><div style="font-size:11px;color:#555;margin-bottom:12px;letter-spacing:1px">PUNTOS ORDENADOS POR {ax}</div>'
        f'<table style="width:100%;font-size:15px;border-collapse:collapse"><tr style="background:#161625;color:#555;font-size:11px">'
        f'<th style="padding:10px 20px;text-align:left">ÍNDICE</th><th style="padding:10px 20px;text-align:left">PUNTO</th><th></th></tr>{rows}</table></div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px">'
        f'<div style="background:{bg};border:1px solid {col};border-radius:8px;padding:20px;text-align:center">'
        f'<div style="font-size:11px;color:{tc};opacity:0.8;margin-bottom:8px;letter-spacing:1px">NODO INSERTADO</div>'
        f'<div style="font-size:28px;font-weight:bold;color:{tc}">{step["median"]}</div></div>'
        f'<div style="background:#07060f;border:1px solid rgba(255,255,255,0.1);border-radius:8px;padding:20px">'
        f'<div style="font-size:11px;color:#555;margin-bottom:8px;letter-spacing:1px">DIVISIÓN RECURSIVA</div>'
        f'<div style="color:#aaa;font-size:14px;line-height:1.8">Izq: {lft}<br>Der: {rgt}</div></div></div></div>')

CSS = """<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Bebas+Neue&display=swap');
.widget-button{ font-family:'Space Mono',monospace!important; font-size:14px!important; letter-spacing:1px!important; border-radius:6px!important; }
.widget-textarea textarea{ font-family:'Space Mono',monospace!important; font-size:16px!important; line-height:1.5!important; }
.widget-vbox, .widget-hbox{ background:#07060f!important; }
.jp-OutputArea-output { background: #07060f !important; }
</style>"""

class KDApp:
    def __init__(self):
        self.steps, self.current = [], 0
        display(HTML(CSS))
        self._build()

    def _build(self):
        hdr = widgets.HTML('<div style="padding:35px 30px;background:#07060f;border-bottom:1px solid #222">'
                           '<div style="font-family:Space Mono;font-size:14px;color:#7eb8ff;letter-spacing:5px;opacity:0.7">VISUALIZADOR AVANZADO</div>'
                           '<div style="font-family:Bebas Neue;font-size:60px;color:#eee;line-height:1">KD-TREE <span style="color:#BA7517">PASO A</span><span style="color:#0F6E56"> PASO</span></div></div>')

        self.inp = widgets.Textarea(value='(2,8), (3,5), (6,1), (9,2), (5,4), (9,6)', layout=widgets.Layout(width='100%', height='80px'))
        self.hint = widgets.HTML('<div style="font-family:Space Mono;font-size:14px;color:#555;margin-top:10px">Formato: (x,y), (x,y)... | Soporta 2D, 3D y más</div>')
        self.inp.observe(self._validate, names='value')

        b_build = widgets.Button(description='CONSTRUIR ÁRBOL', layout=widgets.Layout(width='200px', height='45px'))
        b_reset = widgets.Button(description='REINICIAR',  layout=widgets.Layout(width='150px', height='45px'))
        b_build.on_click(self._build_tree); b_reset.on_click(lambda _: self._reset())

        self.lbl = widgets.HTML(self._lbl())
        self.b_prv = widgets.Button(description='ANTERIOR', disabled=True, layout=widgets.Layout(width='130px', height='40px'))
        self.b_nxt = widgets.Button(description='SIGUIENTE', disabled=True, layout=widgets.Layout(width='130px', height='40px'))
        self.b_aut = widgets.Button(description='AUTO ▶', disabled=True, layout=widgets.Layout(width='100px', height='40px'))
        self.b_prv.on_click(lambda _: self._go(self.current - 1))
        self.b_nxt.on_click(lambda _: self._go(self.current + 1))
        self.b_aut.on_click(self._auto)

        self.out_tree = widgets.HTML(self._empty('Haz clic en CONSTRUIR para visualizar el árbol'))
        self.out_detail = widgets.HTML(self._empty('Los detalles de cada partición aparecerán aquí'))

        panel = widgets.VBox([
            hdr,
            widgets.VBox([
                widgets.HTML('<div style="font-family:Space Mono;font-size:12px;color:#333;margin-bottom:8px;letter-spacing:2px">PUNTOS DE ENTRADA</div>'),
                self.inp, self.hint,
                widgets.HBox([b_build, b_reset], layout=widgets.Layout(gap='15px', margin='15px 0'))
            ], layout=widgets.Layout(padding='30px')),
            widgets.HBox([self.b_prv, self.b_nxt, self.b_aut, self.lbl], layout=widgets.Layout(padding='10px 30px', gap='15px', align_items='center')),
            widgets.HTML('<div style="padding:20px 30px 10px;font-size:12px;color:#333;letter-spacing:3px;font-weight:bold">REPRESENTACIÓN DEL ÁRBOL</div>'),
            widgets.Box([self.out_tree], layout=widgets.Layout(padding='0 30px 30px', width='100%')),
            widgets.HTML('<div style="padding:10px 30px 10px;font-size:12px;color:#333;letter-spacing:3px;font-weight:bold">DETALLE TÉCNICO</div>'),
            widgets.Box([self.out_detail], layout=widgets.Layout(padding='0 30px 40px', width='100%'))
        ], layout=widgets.Layout(background='#07060f', width='100%', border='1px solid #333', border_radius='0px'))

        display(panel)

    def _lbl(self, cur=0, tot=0):
        if tot == 0: return '<span style="font-family:Space Mono;font-size:16px;color:#444;margin-left:20px">0 / 0</span>'
        return f'<span style="font-family:Space Mono;font-size:18px;color:#7eb8ff;font-weight:bold;margin-left:20px">PASO {cur+1} de {tot}</span>'

    def _empty(self, msg):
        return f'<div style="background:#0d0d1a;border:1px solid #222;border-radius:10px;padding:60px;text-align:center;font-family:Space Mono;font-size:16px;color:#444;letter-spacing:1px">{msg}</div>'

    def _validate(self, change):
        try:
            pts = [list(ast.literal_eval(t)) for t in re.findall(r'\([^)]+\)', change['new'])]
            if pts: self.hint.value = f'<div style="color:#0F6E56;font-size:14px;font-weight:bold">{len(pts)} puntos listos para procesar</div>'
        except: self.hint.value = '<div style="color:#A32D2D;font-size:14px">Formato incorrecto. Ejemplo: (1,2), (3,4)</div>'

    def _build_tree(self, _):
        try:
            pts = [list(ast.literal_eval(t)) for t in re.findall(r'\([^)]+\)', self.inp.value)]
            if not pts: return
            _, self.steps = build_kd_logged(pts)
            self.current = 0
            self.b_prv.disabled, self.b_nxt.disabled, self.b_aut.disabled = True, len(self.steps) <= 1, len(self.steps) <= 1
            self._render(0)
        except Exception as e: self.hint.value = f'Error en construcción: {e}'

    def _go(self, i):
        self.current = max(0, min(i, len(self.steps) - 1))
        self.b_prv.disabled, self.b_nxt.disabled = (self.current == 0), (self.current == len(self.steps) - 1)
        self._render(self.current)

    def _auto(self, _):
        def run():
            for i in range(self.current, len(self.steps)):
                self._go(i); time.sleep(1.2)
        threading.Thread(target=run, daemon=True).start()

    def _render(self, i):
        s = self.steps[i]
        svg = tree_svg(build_partial(self.steps, i), highlight=s['median'])
        self.out_tree.value = f'<div style="background:#0d0d1a;border:1px solid #222;border-radius:10px;padding:30px;overflow-x:auto;width:100%">{svg}</div>'
        self.out_detail.value = step_html(s)
        self.lbl.value = self._lbl(i, len(self.steps))

    def _reset(self):
        self.steps, self.current = [], 0
        for b in (self.b_prv, self.b_nxt, self.b_aut): b.disabled = True
        self.lbl.value, self.out_tree.value, self.out_detail.value = self._lbl(), self._empty('Haz clic en CONSTRUIR para visualizar el árbol'), self._empty('...')

KDApp()